In [ ]:
# Loading stuff

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import MinMaxScaler
%matplotlib inline

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.linear_model import Lasso, ElasticNet

In [ ]:
# --- Loading all the datasets
# Train dataset
train_dataset = pd.read_csv('data/train_radiomics_hipocamp.csv')
# Test dataset
test_dataset = pd.read_csv('data/test_radiomics_hipocamp.csv')

In [ ]:
# --- Exploring the train dataset
train_dataset.head()
train_dataset.columns
train_dataset.info(verbose=True, show_counts=True)

In [ ]:
# --- Removing columns with irrelevant information

# Getting all columns whose value is an object
obj_col = train_dataset.select_dtypes(include='object').columns

# Storing in a list the columns that have the same number of unique values as the number of rows
# This means that the column is a unique identifier and probably should be dropped
unique_cols = []
for col in obj_col:
    if train_dataset[col].nunique() == 305:
        unique_cols.append(col)

# list of columns that will be dropped
print(unique_cols)

# dataset with the columns that will be dropped
train_dataset[unique_cols].head()

# The following features might actually be useful:
# - diagnostics_Mask-original_BoundingBox
# - diagnostics_Mask-original_CenterOfMassIndex
# - diagnostics_Mask-original_CenterOfMass

# removing last 3 columns from the 'unique_cols' - we don't want to drop them
unique_cols.remove('diagnostics_Mask-original_BoundingBox')
unique_cols.remove('diagnostics_Mask-original_CenterOfMassIndex')
unique_cols.remove('diagnostics_Mask-original_CenterOfMass')
        
# Dropping the columns stored in the unique_cols list
train_dataset.drop(unique_cols, axis=1, inplace=True)
# Dropping them from the test dataset as well
test_dataset.drop(unique_cols, axis=1, inplace=True)

pd.set_option('display.max_colwidth', None)
display(train_dataset['diagnostics_Mask-original_BoundingBox'].head())
display(train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].head())
display(train_dataset['diagnostics_Mask-original_CenterOfMass'].head())

In [ ]:
# Check if 'diagnostics_Mask-original_CenterOfMassIndex' and 'diagnostics_Mask-original_CenterOfMass' are the same
# If they are, we can drop one of them

(train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] == train_dataset['diagnostics_Mask-original_CenterOfMass']).all()

# They are the same, so we can drop one of them
train_dataset.drop('diagnostics_Mask-original_CenterOfMass', axis=1, inplace=True)

In [ ]:
# --- Transfoming the 'diagnostics_Mask-original_BoundingBox' feature into useful data
# Splitting the string into 6 different columns
# probably x,y,z,width,height,depth -> most common

train_dataset['diagnostics_Mask-original_BoundingBox'] = train_dataset['diagnostics_Mask-original_BoundingBox'].str.replace('(', '').str.replace(')', '')
train_dataset['diagnostics_Mask-original_BoundingBox'] = train_dataset['diagnostics_Mask-original_BoundingBox'].str.split(',').apply(lambda x: [int(i) for i in x])
train_dataset[['x', 'y', 'z', 'width', 'height', 'depth']] = pd.DataFrame(train_dataset['diagnostics_Mask-original_BoundingBox'].tolist(), index=train_dataset.index)
train_dataset.drop('diagnostics_Mask-original_BoundingBox', axis=1, inplace=True)

# Doing the same for the test dataset
test_dataset['diagnostics_Mask-original_BoundingBox'] = test_dataset['diagnostics_Mask-original_BoundingBox'].str.replace('(', '').str.replace(')', '')
test_dataset['diagnostics_Mask-original_BoundingBox'] = test_dataset['diagnostics_Mask-original_BoundingBox'].str.split(',').apply(lambda x: [int(i) for i in x])
test_dataset[['x', 'y', 'z', 'width', 'height', 'depth']] = pd.DataFrame(test_dataset['diagnostics_Mask-original_BoundingBox'].tolist(), index=test_dataset.index)
test_dataset.drop('diagnostics_Mask-original_BoundingBox', axis=1, inplace=True)

# Calculating the volume of the bounding box
train_dataset['volume'] = train_dataset['width'] * train_dataset['height'] * train_dataset['depth']
test_dataset['volume'] = test_dataset['width'] * test_dataset['height'] * test_dataset['depth']

In [ ]:
# --- Transfoming the 'diagnostics_Mask-original_CenterOfMassIndex' feature into useful data
# Splitting the string into 3 different columns

# Removing '(' and ')' from the string
train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.replace('(', '').str.replace(')', '')
train_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.split(',').apply(lambda x: [float(i) for i in x])
train_dataset[['x_center', 'y_center', 'z_center']] = pd.DataFrame(train_dataset['diagnostics_Mask-original_CenterOfMassIndex'].tolist(), index=train_dataset.index)
train_dataset.drop('diagnostics_Mask-original_CenterOfMassIndex', axis=1, inplace=True)

# Doing the same for the test dataset
test_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.replace('(', '').str.replace(')', '')
test_dataset['diagnostics_Mask-original_CenterOfMassIndex'] = test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].str.split(',').apply(lambda x: [float(i) for i in x])
test_dataset[['x_center', 'y_center', 'z_center']] = pd.DataFrame(test_dataset['diagnostics_Mask-original_CenterOfMassIndex'].tolist(), index=test_dataset.index)
test_dataset.drop('diagnostics_Mask-original_CenterOfMassIndex', axis=1, inplace=True)

# Calculating the distance from the center of mass to the bounding box
train_dataset['distance'] = np.sqrt((train_dataset['x'] - train_dataset['x_center'])**2 + (train_dataset['y'] - train_dataset['y_center'])**2 + (train_dataset['z'] - train_dataset['z_center'])**2)
test_dataset['distance'] = np.sqrt((test_dataset['x'] - test_dataset['x_center'])**2 + (test_dataset['y'] - test_dataset['y_center'])**2 + (test_dataset['z'] - test_dataset['z_center'])**2)

In [ ]:
train_dataset.shape

In [ ]:
# --- Dropping constant features
train_dataset = train_dataset.loc[:, (train_dataset != train_dataset.iloc[0]).any()]

# Dropping the same columns on the test dataset
test_dataset = test_dataset[train_dataset.drop(columns='Transition').columns]

In [ ]:
# --- Checking all unique values in the Transition column
train_dataset['Transition'].value_counts()
# Dataset is extremely UNBALANCED

In [ ]:
# --- Using encoding to transform the target columns into numeric values
replace_map = {'Transition': {'CN-CN': 0, 'AD-AD': 1, 'CN-MCI': 2, 'MCI-AD': 3, 'MCI-MCI': 4}}
train_dataset.replace(replace_map, inplace=True)
test_dataset.replace(replace_map, inplace=True)
train_dataset.head()

In [ ]:
train_dataset['Transition'].value_counts()

In [ ]:
train_dataset.info()

In [ ]:
# --- Droppig non-target columns with the same info (correlation = 1 between them)

target_column = 'Transition'  
# Calculating the correlation matrix (excluding the target column)
train_dataset_tmp = train_dataset 
corr_matrix_train = train_dataset_tmp.drop(columns=[target_column]).corr().abs()

# Upper triangle matrix of correlations
upper = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Finding index of feature columns with correlation equal to 1
threshold = 1.0
to_drop = [column for column in upper.columns if any(upper[column] >= threshold)] # for some reason there are columns with correlation > 1.0
print("dropping " + str(len(to_drop)) + " columns")

# Dropping the columns
train_dataset = train_dataset.drop(columns=to_drop)
test_dataset = test_dataset.drop(columns=to_drop)

In [ ]:
# Checking dataset info
train_dataset.info()

In [ ]:
## --- Dropping columns with low correlation with the target column (<0.005)

#corr_matrix = train_dataset.corr()
#corr_target = corr_matrix['Transition'].abs()
#uncorr_cols = []
#for i in range(len(corr_target)):
#    if corr_target[i] < 0.005:
#        uncorr_cols.append(corr_matrix.columns[i])
#
## Dropping the columns stored in the uncorr_cols list
#n_cols_before = len(train_dataset.columns)
#train_dataset.drop(uncorr_cols, axis=1, inplace=True)
#test_dataset.drop(uncorr_cols, axis=1, inplace=True)
#n_cols_after = len(train_dataset.columns)
#
#print("Dropped " + str(n_cols_before-n_cols_after) + " columns!")

In [ ]:
# --- Checking Age column
print(train_dataset['Age'].hist())
plt.show()
# Not a normal distribution

In [ ]:
# --- Checking Sex column
print(train_dataset['Sex'].value_counts(normalize=True))
# Not quite balanced

In [ ]:
# --- Understanding the relationship between Age and Transition
sns.catplot(x="Transition", y="Age", data=train_dataset, kind="box", aspect=1.5)
plt.title("Boxplot for Transition vs Age")
plt.show()

In [ ]:
# --- Treating outliers
# Outliers should be converted to the closest bound

changes_count = 0

for col in train_dataset.columns:
    if col not in ['Transition', 'ID']:
        median = train_dataset[col].median()
        q1 = train_dataset[col].quantile(0.25)
        q3 = train_dataset[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        # Count outliers before replacing them
        changes_count += ((train_dataset[col] < lower_bound) | (train_dataset[col] > upper_bound)).sum()
        changes_count += ((test_dataset[col] < lower_bound) | (test_dataset[col] > upper_bound)).sum()
        
        # Replace outliers with the median
        # train_dataset[col] = train_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)
        # test_dataset[col] = test_dataset[col].apply(lambda x: median if x < lower_bound or x > upper_bound else x)

        # Replace outliers with the closest bound
        train_dataset[col] = train_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)
        test_dataset[col] = test_dataset[col].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)

print(f"Total number of changes made: {changes_count}")

In [ ]:
X = train_dataset.drop(columns=['Transition'])
y = train_dataset['Transition']

In [ ]:
# --- Standardizing data

scaler = StandardScaler()
X = scaler.fit_transform(X)
X = pd.DataFrame(X, columns=train_dataset.drop(columns=['Transition']).columns)

test_dataset = scaler.transform(test_dataset)  # Use same scaler for test data
test_dataset = pd.DataFrame(test_dataset, columns=train_dataset.drop(columns=['Transition']).columns)

In [ ]:
# Add an 'oversampled' column to the dataset with 'NO' values
# This will be used to guarantee oversampling data is not used in testing
train_dataset['oversampled'] = 'NO'

In [ ]:
# --- Oversampling data
### Oversampling should be done after standardizing but before feature selection and dimensionality reduction

# Split the dataset into features and target
X = train_dataset.drop(columns=['Transition', 'oversampled'])
y = train_dataset['Transition']

# Identify the majority class count (class '0')
majority_class_count = y.value_counts().loc[0]

# Set the target count for class '2' to be 0.5 of the majority class count
target_class_2_count = int(majority_class_count * 0.3)
target_class_1_count = int(majority_class_count * 1)
target_class_3_count = int(majority_class_count * 1)
target_class_4_count = int(majority_class_count * 1)

# Define a sampling strategy where only class '2' is oversampled to the target count
sampling_strategy = {2: target_class_2_count,
#                     1: target_class_1_count,
#                     3: target_class_3_count,
#                     4: target_class_4_count
}

# Apply SMOTE with the defined strategy
smote = SMOTE(sampling_strategy=sampling_strategy, random_state=123)
X_smote, y_smote = smote.fit_resample(X, y)

# Create a new dataframe with the oversampled data
oversampled_data = pd.concat([X_smote, y_smote], axis=1)
oversampled_data.columns = list(X.columns) + ['Transition']

# Mark the oversampled rows with 'YES' in the 'oversampled' column
oversampled_data['oversampled'] = ['YES' if idx >= len(X) else 'NO' for idx in range(len(X_smote))]

# Concatenate the new oversampled data with the original data
train_dataset = pd.concat([train_dataset, oversampled_data[oversampled_data['oversampled'] == 'YES']], ignore_index=True)

# Display the final class distribution for verification
print("Class distribution after SMOTE oversampling:")
print(train_dataset['Transition'].value_counts())

# --- Getting the number of normal and oversampled data
train_dataset['oversampled'].value_counts()

X = train_dataset.drop(columns=['Transition','oversampled'])
y = train_dataset['Transition']

In [ ]:
## --- Feature selection with ANOVA
#
#f, p = f_classif(X, y)
#f_classif_importances = f
#features_to_keep = X.columns[f_classif_importances.argsort()[-200:]]
### Keeping only the selected features
#X = X[list(features_to_keep)]
## Dropping the non-selected columns from the test dataset as well
#test_dataset = test_dataset[features_to_keep]
#print('Number of features:', len(features_to_keep))

In [ ]:
## --- Dimensionality reduction with PCA
#
#pca = PCA(n_components=100) 
#pca.fit(X)
#data = pca.transform(X)
#train_dataset_no_target = pd.DataFrame(data)
#train_dataset = pd.concat([train_dataset_no_target, train_dataset['Transition'], train_dataset['oversampled']], axis=1)
#
#data_test = pca.transform(test_dataset)
#test_dataset = pd.DataFrame(data_test)

In [ ]:
train_dataset.shape

In [ ]:
train_dataset.head()

In [ ]:
# --- Checking feature importance for the created-above features

lasso = Lasso(alpha=0.01)
lasso.fit(X, y)
lasso_importances = lasso.coef_
# order by importance
lasso_importances = np.abs(lasso_importances)
lasso_importances = lasso_importances.argsort()[::-1]
features_to_keep = X.columns[lasso_importances]


# volume, distance, x, y, z, width, height, depth, x_center, y_center, z_center
volume_index = list(X.columns).index('volume')
print(f"Importance of 'Volume': {lasso.coef_[volume_index]}")
# Getting importance ranking 
print(f"Ranking of 'Volume': {np.where(lasso_importances == volume_index)[0][0]}" + " out of " + str(len(lasso_importances)))

distance_index = list(X.columns).index('distance')
print(f"Importance of 'Distance': {lasso.coef_[distance_index]}")
# Getting importance ranking
print(f"Ranking of 'Distance': {np.where(lasso_importances == distance_index)[0][0]}" + " out of " + str(len(lasso_importances)))

x_index = list(X.columns).index('x')
print(f"Importance of 'x': {lasso.coef_[x_index]}")
print(f"Ranking of 'x': {np.where(lasso_importances == x_index)[0][0]}" + " out of " + str(len(lasso_importances)))

y_index = list(X.columns).index('y')
print(f"Importance of 'y': {lasso.coef_[y_index]}")
print(f"Ranking of 'y': {np.where(lasso_importances == y_index)[0][0]}" + " out of " + str(len(lasso_importances)))

z_index = list(X.columns).index('z')
print(f"Importance of 'z': {lasso.coef_[z_index]}")
print(f"Ranking of 'z': {np.where(lasso_importances == z_index)[0][0]}" + " out of " + str(len(lasso_importances)))

width_index = list(X.columns).index('width')
print(f"Importance of 'width': {lasso.coef_[width_index]}")
print(f"Ranking of 'width': {np.where(lasso_importances == width_index)[0][0]}" + " out of " + str(len(lasso_importances)))

height_index = list(X.columns).index('height')
print(f"Importance of 'height': {lasso.coef_[height_index]}")
print(f"Ranking of 'height': {np.where(lasso_importances == height_index)[0][0]}" + " out of " + str(len(lasso_importances)))

depth_index = list(X.columns).index('depth')
print(f"Importance of 'depth': {lasso.coef_[depth_index]}")
print(f"Ranking of 'depth': {np.where(lasso_importances == depth_index)[0][0]}" + " out of " + str(len(lasso_importances)))

x_center_index = list(X.columns).index('x_center')
print(f"Importance of 'x_center': {lasso.coef_[x_center_index]}")
print(f"Ranking of 'x_center': {np.where(lasso_importances == x_center_index)[0][0]}" + " out of " + str(len(lasso_importances)))

y_center_index = list(X.columns).index('y_center')
print(f"Importance of 'y_center': {lasso.coef_[y_center_index]}")
print(f"Ranking of 'y_center': {np.where(lasso_importances == y_center_index)[0][0]}" + " out of " + str(len(lasso_importances)))

z_center_index = list(X.columns).index('z_center')
print(f"Importance of 'z_center': {lasso.coef_[z_center_index]}")
print(f"Ranking of 'z_center': {np.where(lasso_importances == z_center_index)[0][0]}" + " out of " + str(len(lasso_importances)))



In [ ]:
# --- Saving the treated datasets
train_dataset.to_csv('data/train_dataset.csv', index=False)
test_dataset.to_csv('data/test_dataset.csv', index=False)